# Анализ оттока клиентов (Churn) с использованием SQL

## Обзор проекта

В данном ноутбуке демонстрируется использование SQL для аналитики данных в задаче оттока клиентов телеком-провайдера.

Цели анализа:
- Очистка и подготовка данных с использованием SQL
- Проведение EDA (разведочного анализа) через SQL-запросы
- Выявление факторов, влияющих на отток клиентов
- Демонстрация интеграции SQL и Python (pandas)

Датасет: Telco Customer Churn (~7000 клиентов, 21 признак)

## Настройка базы данных

Для выполнения SQL-запросов используется SQLite — лёгкая встроенная СУБД, которая удобно интегрируется с Python через SQLAlchemy.

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///../data/telecom.db")

## Загрузка данных

Исходные данные загружаются из CSV-файла и сохраняются в SQL-базу для дальнейшего анализа.

In [2]:
df = pd.read_csv("../data/Telco_Churn_Dataset.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.to_sql("telecom_customers_raw", engine, if_exists="replace", index=False)

7043

## Первичный анализ данных

Проверяем структуру таблицы и корректность загрузки данных.

In [4]:
query = """
SELECT *
FROM telecom_customers_raw
LIMIT 5
"""
pd.read_sql(query, engine)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Очистка и подготовка данных (Data Cleaning)

На данном этапе выполняются ключевые преобразования:

- Приведение числовых признаков к корректным типам
- Обработка пропущенных значений (TotalCharges)
- Преобразование целевой переменной Churn в бинарный формат (0/1)
- Приведение названий столбцов к единому стилю

Данный этап является критически важным, так как качество данных напрямую влияет на результаты анализа и моделей.

In [5]:
query = """
CREATE TABLE telecom_customers_clean AS
SELECT
    customerID AS customer_id,
    gender,
    SeniorCitizen AS senior_citizen,
    Partner AS partner,
    Dependents AS dependents,
    tenure,
    PhoneService AS phone_service,
    MultipleLines AS multiple_lines,
    InternetService AS internet_service,
    OnlineSecurity AS online_security,
    OnlineBackup AS online_backup,
    DeviceProtection AS device_protection,
    TechSupport AS tech_support,
    StreamingTV AS streaming_tv,
    StreamingMovies AS streaming_movies,
    Contract AS contract,
    PaperlessBilling AS paperless_billing,
    PaymentMethod AS payment_method,

    CAST(MonthlyCharges AS REAL) AS monthly_charges,

    CASE 
        WHEN TotalCharges = '' THEN NULL
        ELSE CAST(TotalCharges AS REAL)
    END AS total_charges,

    CASE 
        WHEN Churn = 'Yes' THEN 1
        ELSE 0
    END AS churn

FROM telecom_customers_raw
"""
with engine.connect() as conn:
    result = conn.execute(text(query))
    print(f"Новая таблица успешно создана")

Новая таблица успешно создана


## Feature Engineering в SQL

Дополнительно создаются новые признаки:

- num_services — количество подключённых услуг
- tenure_segment — сегментация клиентов по длительности использования

Это позволяет проводить более глубокий анализ поведения клиентов.

In [6]:
query = """
CREATE TABLE telecom_features AS
SELECT
    *,

    (
        (online_security = 'Yes') +
        (online_backup = 'Yes') +
        (device_protection = 'Yes') +
        (tech_support = 'Yes') +
        (streaming_tv = 'Yes') +
        (streaming_movies = 'Yes')
    ) AS num_services,

    CASE
        WHEN tenure <= 12 THEN '0-12'
        WHEN tenure <= 24 THEN '13-24'
        WHEN tenure <= 48 THEN '25-48'
        ELSE '49+'
    END AS tenure_segment

FROM telecom_customers_clean
"""
with engine.connect() as conn:
    conn.execute(text(query))
    print("Новая таблица успешно создана")
    

Новая таблица успешно создана


## Общий уровень оттока клиентов

Рассчитаем базовую метрику churn rate — долю клиентов, прекративших использование услуг.

In [7]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) AS churn_rate
FROM telecom_features
"""
pd.read_sql(query, engine)

,total_customers,churned,churn_rate
0,7043,1869,26.54


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[12], line 24
      1 query = """
      2 CREATE TABLE telecom_features AS
      3 SELECT
   (...)
     22 FROM telecom_customers_clean
     23 """
---> 24 engine.execute(query)

AttributeError: 'Engine' object has no attribute 'execute'

**Результат:**
- Всего клиентов: 7043
- Ушедших клиентов: 1869
- Уровень оттока: 26.54%

**Интерпретация:**

- Примерно каждый четвёртый клиент прекращает пользоваться услугами компании
- Это достаточно высокий уровень оттока, требующий анализа причин
- Данная метрика используется как базовая точка отсчёта для дальнейшего сравнения сегментов

## Отток в зависимости от типа контракта

In [8]:
query = """
SELECT
    contract,
    COUNT(*) AS total,
    SUM(churn) AS churned,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) AS churn_rate
FROM telecom_features
GROUP BY contract
ORDER BY churn_rate DESC
"""
pd.read_sql(query, engine)

,contract,total,churned,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


**Результаты:**

- Month-to-month: 42.71%
- One year: 11.27%
- Two year: 2.83%

**Интерпретация:**

- Клиенты с помесячным контрактом имеют **крайне высокий уровень оттока (42.7%)**
- Долгосрочные контракты значительно снижают churn:
  - годовой контракт снижает отток почти в 4 раза
  - двухлетний — более чем в 15 раз

**Вывод:**

- Тип контракта — один из **самых сильных факторов оттока**
- Клиенты без долгосрочных обязательств легче принимают решение об уходе
- Это потенциальная точка роста: стимулирование перехода на долгосрочные тарифы

## Анализ высокорисковых сегментов

Рассмотрим комбинации факторов (контракт, интернет, способ оплаты) для выявления сегментов с максимальным риском оттока.

In [9]:
query = """
SELECT
    contract,
    internet_service,
    payment_method,
    COUNT(*) AS total,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) AS churn_rate
FROM telecom_features
GROUP BY contract, internet_service, payment_method
HAVING COUNT(*) >= 200
ORDER BY churn_rate DESC
LIMIT 10
"""
pd.read_sql(query, engine)

,contract,internet_service,payment_method,total,churn_rate
0,Month-to-month,Fiber optic,Electronic check,1307,60.37
1,Month-to-month,Fiber optic,Mailed check,201,50.75
2,Month-to-month,Fiber optic,Bank transfer (automatic),327,45.57
3,Month-to-month,Fiber optic,Credit card (automatic),293,41.64
4,Month-to-month,DSL,Electronic check,474,40.51
5,Month-to-month,DSL,Mailed check,367,30.79
6,Month-to-month,No,Mailed check,325,20.62
7,Two year,DSL,Credit card (automatic),237,2.11
8,Two year,DSL,Bank transfer (automatic),224,1.34
9,Two year,No,Mailed check,252,0.79


**Ключевые наблюдения:**

ТОП сегменты по оттоку:

1. Month-to-month + Fiber optic + Electronic check → **60.37%**
2. Month-to-month + Fiber optic + Mailed check → 50.75%
3. Month-to-month + Fiber optic + Bank transfer → 45.57%

Минимальный отток:

- Two year + DSL + Mailed check → **0.79%**
- Two year + DSL + Bank transfer → 1.34%

**Интерпретация:**

- Наиболее рискованный профиль клиента:
  - краткосрочный контракт
  - дорогой интернет (Fiber)
  - "менее стабильные" способы оплаты (особенно electronic check)

- Наиболее стабильные клиенты:
  - долгосрочный контракт
  - более дешёвые или стабильные услуги (DSL)
  - автоматические или традиционные способы оплаты

**Вывод:**

- Отток определяется не одним фактором, а **комбинацией признаков**
- Это критично для построения ML-модели (нелинейные зависимости)
- Можно использовать для таргетированных retention-стратегий

## Влияние количества подключённых услуг

In [10]:
query = """
SELECT
    num_services,
    COUNT(*) AS total,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) AS churn_rate
FROM telecom_features
GROUP BY num_services
ORDER BY num_services
"""
pd.read_sql(query, engine)

,num_services,total,churn_rate
0,0,2219,21.41
1,1,966,45.76
2,2,1033,35.82
3,3,1118,27.37
4,4,852,22.30
5,5,571,12.43
6,6,284,5.28


**Результаты:**

- 0 услуг → 21.41%
- 1 услуга → 45.76% (пик)
- 6 услуг → 5.28%

**Интерпретация:**

- Наблюдается **нелинейная зависимость**
- Клиенты с 1–2 услугами имеют самый высокий churn
- Клиенты с большим количеством услуг значительно более лояльны

**Вывод:**

- Чем глубже клиент "встроен" в экосистему сервиса, тем ниже вероятность ухода
- Это классический эффект "lock-in"
- Потенциальная стратегия: upsell дополнительных услуг

## Отток в зависимости от длительности пользования

In [11]:
query = """
SELECT
    tenure_segment,
    COUNT(*) AS total,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) AS churn_rate
FROM telecom_features
GROUP BY tenure_segment
ORDER BY churn_rate DESC
"""
pd.read_sql(query, engine)

,tenure_segment,total,churn_rate
0,0-12,2186,47.44
1,13-24,1024,28.71
2,25-48,1594,20.39
3,49+,2239,9.51


**Результаты:**

- 0–12 месяцев → 47.44%
- 13–24 → 28.71%
- 25–48 → 20.39%
- 49+ → 9.51%

**Интерпретация:**

- Максимальный churn наблюдается у новых клиентов
- По мере увеличения срока использования:
  - churn стабильно снижается
  - формируется лояльность

**Вывод:**

- Критический период — первый год
- Основные потери происходят на раннем этапе
- Это указывает на проблемы:
  - onboarding
  - соответствие ожиданиям
  - качество сервиса в начале

**Бизнес-следствие:**

- необходимо фокусироваться на удержании новых клиентов

## Ключевые выводы

В ходе анализа с использованием SQL были выявлены основные факторы оттока:

1. Тип контракта — самый сильный фактор (до 42% churn)
2. Длительность использования — критический фактор (до 47% у новых клиентов)
3. Количество услуг — влияет на удержание (до 5% у "глубоких" клиентов)
4. Комбинации факторов дают экстремальные значения churn (до 60%)

## Заключение

SQL был использован для:
- очистки данных
- feature engineering
- сегментации клиентов
- выявления ключевых факторов оттока

Данный подход может быть напрямую интегрирован в ML pipeline для построения моделей предсказания оттока.